# Advanced tutorial: Discover stations and observations

## Research question

**Which Virginia candidate stations list both water temperature and dissolved oxygen, also have streamflow, and return approved observations suitable for analysis?**

## `streamwq` functions used

- `stream.catalog.DATASETS`
- `stream.get_sites()`
- `stream.get_data("water_quality", ...)`
- `summarize_observation_period()`

## 1. Discover datasets and variables

In [ ]:
import streamwq as stream
from streamwq.helpers.pandas import summarize_observation_period

print(stream.catalog.DATASETS)
print(stream.catalog.VARIABLES["water_quality"])

## 2. Find candidate stations

Discovery returns candidate IDs from metadata; alternatives within the water-quality variable filter are combined as OR, so this example checks metadata to retain stations listing both variables. Candidate availability does not guarantee records for the later quality filter.

In [ ]:
candidate_sites = stream.get_sites(
    states="Virginia",
    variables=["WTemp_C", "DO_mgL"],
    streamflow=True,
)
candidate_metadata = stream.get_metadata(site_ids=candidate_sites).to_pandas()
required_variables = {"WTemp_C", "DO_mgL"}
available_variables = candidate_metadata["WQ_parameters"].fillna("").map(
    lambda values: {value.strip() for value in values.split(",") if value.strip()}
)
site_metadata = candidate_metadata.loc[
    available_variables.map(required_variables.issubset)
].reset_index(drop=True)
if site_metadata.empty:
    raise ValueError("No Virginia candidate reports both requested water-quality variables")
site_ids = site_metadata["STREAM_ID"].tolist()
print(f"Found {len(site_ids):,} candidate stations with both variables")
site_metadata[["STREAM_ID", "site_name", "state_name"]].head(10)

## 3. Retrieve approved observations

Begin with a few stations while developing an analysis. 

In [ ]:
%%time

water_quality = stream.get_data(
    "water_quality",
    site_ids=site_ids,
    columns=["STREAM_ID", "DateTime", "WTemp_C", "DO_mgL"],
    quality="approved",
).to_pandas()
if water_quality.empty:
    raise ValueError(
        "These candidate sites have no approved water-quality rows; "
        "discovery does not guarantee observations. Broaden the filters or date range."
    )
water_quality.head()

## 4. Identify the available observation periods

In [ ]:
periods = summarize_observation_period(water_quality)
periods.sort_values("duration_days", ascending=False)